# AI-Gated Markowitz Portfolio - Year 2025 Test (Multi-Sheet)
This notebook runs only the AI-Gated Markowitz strategy on 2025 test data and saves:
- Sheet 1 (Summary): Clean view with only significant allocations (> 0.01%)
- Sheet 2 (Detail): Complete data with all values

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
from scipy.optimize import minimize
import os

print('Libraries imported successfully')

Libraries imported successfully


In [2]:
# Load Multi-Asset Data
file_path = 'real_crypto_data.csv'
df = pd.read_csv(file_path, parse_dates=['Date'], index_col='Date')
print("Loaded Assets:", df.columns.tolist())

def get_market_features(df):
    returns = df.pct_change().fillna(0)
    volatility = returns.rolling(window=20).std().fillna(0)
    sma50 = df.rolling(window=50).mean()
    momentum = (df / sma50 - 1).fillna(0)
    return returns, volatility, momentum

returns_df, vol_df, mom_df = get_market_features(df)
assets = df.columns.tolist()
n_assets = len(assets)

# Define Training Range (2023 and 2024)
train_mask = (df.index.year == 2023) | (df.index.year == 2024)
train_df_slice = df[train_mask]
train_returns_slice = returns_df[train_mask]

# Define Test Range (2025 Only)
test_mask = (df.index.year == 2025)
test_df_slice = df[test_mask]
test_returns_slice = returns_df[test_mask]

print(f"Total Data Points: {len(df)}")
print(f"Training Data (2023-2024): {len(train_df_slice)} days")
print(f"Testing Data (2025): {len(test_df_slice)} days")

Loaded Assets: ['BCH', 'BTC', 'ETH', 'LTC', 'XRP']
Total Data Points: 1096
Training Data (2023-2024): 731 days
Testing Data (2025): 365 days


In [3]:
# Train AI Gatekeeper (Bull/Bear Classifier)
print('Training AI Gatekeeper (Bull/Bear Classifier)...')
market_ret = returns_df.mean(axis=1)
feat = pd.DataFrame(index=returns_df.index)
feat['Vol_20'] = market_ret.rolling(20).std()
feat['Mom_20'] = market_ret.rolling(20).mean()
feat['Target'] = (market_ret.shift(-1) > 0).astype(int)
feat = feat.dropna()

X_rf = feat[['Vol_20', 'Mom_20']]
y_rf = feat['Target']

# Train on 2023-2024 data only
train_rf_mask = (feat.index.year == 2023) | (feat.index.year == 2024)
X_train_rf = X_rf[train_rf_mask]
y_train_rf = y_rf[train_rf_mask]

clf = RandomForestClassifier(n_estimators=100, min_samples_split=10, random_state=42)
clf.fit(X_train_rf, y_train_rf)

# Predict on ALL data
rf_probs_all = clf.predict_proba(X_rf)[:, 1]
rf_probs_series = pd.Series(rf_probs_all, index=X_rf.index)
print(f'AI Gatekeeper Trained. Train Acc: {accuracy_score(y_train_rf, clf.predict(X_train_rf)):.2f}')

Training AI Gatekeeper (Bull/Bear Classifier)...
AI Gatekeeper Trained. Train Acc: 0.88


In [4]:
# Markowitz Optimization Function
def optimize_markowitz(returns_window, cov_matrix):
    n = returns_window.shape[1]
    if n == 0: 
        return np.array([])
    
    def neg_sharpe(weights):
        portfolio_return = np.sum(returns_window.mean() * weights) * 252
        portfolio_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
        return -(portfolio_return / portfolio_vol) if portfolio_vol > 0 else 0
    
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(n))
    init_guess = n * [1. / n]
    
    try:
        res = minimize(neg_sharpe, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
        return res.x
    except:
        return np.array(init_guess)

print('Markowitz optimization function defined')

Markowitz optimization function defined


In [5]:
# Run AI-Gated Markowitz Strategy on 2025 Test Data
print(f"\n{'='*10} Running AI-Gated Markowitz on Year 2025 {'='*10}")

# Initialize
initial_capital = 100.0  # Starting with $100
lookback = 50
portfolio_history = []

# Current portfolio value
current_value = initial_capital

# Iterate through 2025 test data
for step in range(len(test_df_slice) - 1):
    current_date = test_df_slice.index[step]
    next_date = test_df_slice.index[step + 1]
    
    # Get current prices
    current_prices = test_df_slice.iloc[step].values
    next_returns = test_returns_slice.iloc[step + 1].values
    
    # Calculate Markowitz weights
    glob_idx = returns_df.index.get_loc(current_date)
    if glob_idx >= lookback:
        window_ret = returns_df.iloc[glob_idx-lookback:glob_idx]
        cov_mat = window_ret.cov().values
        mw_weights = optimize_markowitz(window_ret, cov_mat)
    else:
        mw_weights = np.ones(n_assets) / n_assets
    
    # Apply AI Gating
    if current_date in rf_probs_series.index:
        bull_prob = rf_probs_series.loc[current_date]
        if bull_prob > 0.52:
            ai_weights = mw_weights  # Bullish: full allocation
        elif bull_prob < 0.48:
            ai_weights = np.zeros(n_assets)  # Bearish: cash
        else:
            ai_weights = 0.5 * mw_weights  # Neutral: 50% allocation
    else:
        ai_weights = 0.5 * mw_weights
    
    # Calculate dollar allocation for each asset
    dollar_allocation = current_value * ai_weights
    
    # Calculate units of each asset
    units = dollar_allocation / current_prices
    
    # Calculate cash position
    total_invested = np.sum(ai_weights)
    cash_percentage = 1.0 - total_invested
    cash_usd = current_value * cash_percentage
    
    # Calculate portfolio return
    portfolio_return = np.sum(ai_weights * next_returns)
    
    # Update portfolio value
    current_value = current_value * (1 + portfolio_return)
    
    # Store daily portfolio data
    portfolio_data = {
        'Date': current_date,
        'Bull_Probability': rf_probs_series.loc[current_date] * 100 if current_date in rf_probs_series.index else np.nan
    }
    
    # Add units and weights for each asset (weights in percentage)
    for i, asset in enumerate(assets):
        portfolio_data[f'{asset}_units'] = units[i]
        portfolio_data[f'{asset}_weight_%'] = ai_weights[i] * 100
    
    # Add cash information
    portfolio_data['Cash_USD'] = cash_usd
    portfolio_data['Cash_%'] = cash_percentage * 100
    portfolio_data['Portfolio_Value_USD'] = current_value
    portfolio_history.append(portfolio_data)

# Create DataFrame
portfolio_df = pd.DataFrame(portfolio_history)

print(f"\nFinal Portfolio Value: ${current_value:.2f}")
print(f"Total Return: {((current_value - initial_capital) / initial_capital * 100):.2f}%")
print(f"\nPortfolio data collected for {len(portfolio_df)} days")
print(f"Average Cash Position: {portfolio_df['Cash_%'].mean():.2f}%")


========== Running AI-Gated Markowitz on Year 2025 ==========

Final Portfolio Value: $190.93
Total Return: 90.93%

Portfolio data collected for 364 days
Average Cash Position: 35.58%


In [6]:
# Create Summary DataFrame (only significant allocations > 0.01%)
print("\nCreating Summary Sheet (clean view)...")

# Start with basic columns
summary_df = portfolio_df[['Date', 'Bull_Probability', 'Portfolio_Value_USD', 'Cash_USD', 'Cash_%']].copy()

# Add active assets information
threshold = 0.01  # 0.01% threshold

# For each row, create a string of active assets
active_assets_list = []
for idx, row in portfolio_df.iterrows():
    active = []
    for asset in assets:
        weight_pct = row[f'{asset}_weight_%']
        if weight_pct > threshold:
            units = row[f'{asset}_units']
            active.append(f"{asset}: {weight_pct:.2f}% ({units:.4f} units)")
    
    if len(active) == 0:
        active_assets_list.append("100% Cash")
    else:
        active_assets_list.append(" | ".join(active))

summary_df['Active_Allocations'] = active_assets_list

# Reorder columns for better readability
summary_df = summary_df[['Date', 'Bull_Probability', 'Active_Allocations', 'Cash_%', 'Cash_USD', 'Portfolio_Value_USD']]

print(f"Summary sheet created with {len(summary_df)} rows")
print(f"\nFirst 5 rows of Summary:")
print(summary_df.head())


Creating Summary Sheet (clean view)...
Summary sheet created with 364 rows

First 5 rows of Summary:
        Date  Bull_Probability  \
0 2025-01-01         48.686769   
1 2025-01-02         37.511238   
2 2025-01-03         73.080423   
3 2025-01-04         52.476552   
4 2025-01-05         50.976552   

                                  Active_Allocations        Cash_%  \
0                        XRP: 50.00% (21.5300 units)  5.000000e+01   
1                                          100% Cash  1.000000e+02   
2                       XRP: 100.00% (41.4794 units)  2.220446e-14   
3                       XRP: 100.00% (41.4794 units) -4.440892e-14   
4  ETH: 3.53% (0.0010 units) | XRP: 46.47% (19.27...  5.000000e+01   

       Cash_USD  Portfolio_Value_USD  
0  5.000000e+01           101.744553  
1  1.017446e+02           101.744553  
2  2.259183e-14           100.363244  
3 -4.457023e-14            99.539137  
4  4.976957e+01            99.959633  


In [7]:
# Save to Excel with multiple sheets
output_file = 'ai_gated_markowitz_portfolio_2025_multi_sheet.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Sheet 1: Summary (clean view)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Sheet 2: Detail (complete data)
    portfolio_df.to_excel(writer, sheet_name='Detail', index=False)

print(f"\n{'='*60}")
print(f"Portfolio data saved to: {output_file}")
print(f"{'='*60}")
print(f"\nSheet 1 'Summary': Clean view with only significant allocations (> 0.01%)")
print(f"  - Columns: Date, Bull_Probability, Active_Allocations, Cash_%, Cash_USD, Portfolio_Value_USD")
print(f"  - {len(summary_df)} rows")
print(f"\nSheet 2 'Detail': Complete data with all values")
print(f"  - All columns including individual asset weights and units")
print(f"  - {len(portfolio_df)} rows")
print(f"\nColumn names in Detail sheet:")
print(portfolio_df.columns.tolist())


Portfolio data saved to: ai_gated_markowitz_portfolio_2025_multi_sheet.xlsx

Sheet 1 'Summary': Clean view with only significant allocations (> 0.01%)
  - Columns: Date, Bull_Probability, Active_Allocations, Cash_%, Cash_USD, Portfolio_Value_USD
  - 364 rows

Sheet 2 'Detail': Complete data with all values
  - All columns including individual asset weights and units
  - 364 rows

Column names in Detail sheet:
['Date', 'Bull_Probability', 'BCH_units', 'BCH_weight_%', 'BTC_units', 'BTC_weight_%', 'ETH_units', 'ETH_weight_%', 'LTC_units', 'LTC_weight_%', 'XRP_units', 'XRP_weight_%', 'Cash_USD', 'Cash_%', 'Portfolio_Value_USD']


In [8]:
# Display summary statistics
print("\n" + "="*50)
print("PORTFOLIO SUMMARY STATISTICS")
print("="*50)
print(f"\nInitial Capital: ${initial_capital:.2f}")
print(f"Final Portfolio Value: ${current_value:.2f}")
print(f"Total Return: {((current_value - initial_capital) / initial_capital * 100):.2f}%")
print(f"\nPortfolio Value Statistics:")
print(portfolio_df['Portfolio_Value_USD'].describe())

print(f"\nAverage Asset Allocation (by weight %):")
for asset in assets:
    avg_weight = portfolio_df[f'{asset}_weight_%'].mean()
    print(f"{asset}: {avg_weight:.2f}%")

print(f"\nCash Position Statistics:")
print(f"Average Cash: {portfolio_df['Cash_%'].mean():.2f}%")
print(f"Max Cash: {portfolio_df['Cash_%'].max():.2f}%")
print(f"Min Cash: {portfolio_df['Cash_%'].min():.2f}%")
print(f"Days with 100% Cash: {(portfolio_df['Cash_%'] == 100).sum()} days")
print(f"Days with 0% Cash: {(portfolio_df['Cash_%'] == 0).sum()} days")

# Count days with significant allocations per asset
print(f"\nDays with Significant Allocation (> 0.01%):")
for asset in assets:
    days_active = (portfolio_df[f'{asset}_weight_%'] > 0.01).sum()
    print(f"{asset}: {days_active} days ({days_active/len(portfolio_df)*100:.1f}%)")


PORTFOLIO SUMMARY STATISTICS

Initial Capital: $100.00
Final Portfolio Value: $190.93
Total Return: 90.93%

Portfolio Value Statistics:
count    364.000000
mean     149.326265
std       34.303129
min       87.752266
25%      119.291054
50%      148.581369
75%      178.479482
max      212.535382
Name: Portfolio_Value_USD, dtype: float64

Average Asset Allocation (by weight %):
BCH: 18.38%
BTC: 11.02%
ETH: 12.90%
LTC: 8.00%
XRP: 14.13%

Cash Position Statistics:
Average Cash: 35.58%
Max Cash: 100.00%
Min Cash: -0.00%
Days with 100% Cash: 113 days
Days with 0% Cash: 46 days

Days with Significant Allocation (> 0.01%):
BCH: 92 days (25.3%)
BTC: 49 days (13.5%)
ETH: 79 days (21.7%)
LTC: 46 days (12.6%)
XRP: 76 days (20.9%)
